# NYC Collision Severity - Datalake & Athena Setup
This notebook initializes the S3 Datalake and registers the processed NYC Collision data as an Athena table for SQL-based EDA.

In [ ]:
!pip install "sagemaker<3.0.0" awswrangler pandas

import boto3
import sagemaker
import pandas as pd
import awswrangler as wr
import os

# 1. Setup AWS Connections
sess = sagemaker.Session()
bucket = sess.default_bucket()
region = sess.boto_region_name
prefix = "aai-540-group6/nyc-collisions"

# S3 path for the Datalake
s3_path = f"s3://{bucket}/{prefix}/data/raw/"

print(f"Target S3 Datalake Path: {s3_path}")

### Load Merged Sample Data
We use the merged dataset created during our Phase 1 implementation.

In [ ]:
# Path to the merged sample data we pushed to GitHub
local_data_path = '../data/raw/merged_sample.csv'

if os.path.exists(local_data_path):
    df = pd.read_csv(local_data_path)
    print(f"Loaded merged dataset with {len(df)} rows.")
else:
    print("Local data not found. Please ensure Phase 1 data ingestion has run.")

### Register Datalake with Athena
Using AWS Data Wrangler to push the DataFrame to S3 and create the Athena schema automatically.

In [ ]:
database_name = 'nyc_collision_db'
table_name = 'merged_collisions'

# Create the Athena Database
wr.catalog.create_database(database_name, exist_ok=True)

# Upload to S3 and register the table in Athena
wr.s3.to_csv(
    df=df,
    path=s3_path,
    dataset=True,
    database=database_name,
    table=table_name,
    index=False,
    mode='overwrite'
)

print(f"Successfully registered {table_name} in Athena database {database_name}!")

### Initial SQL-based EDA (Athena Queries)
Below are some sample queries to explore the collision data via Athena.

In [ ]:
# Query 1: Distribution of collisions by Borough
query_1 = f"""
SELECT borough, COUNT(*) as collision_count
FROM {database_name}.{table_name}
GROUP BY borough
ORDER BY collision_count DESC
"""
borough_dist = wr.athena.read_sql_query(query_1, database=database_name)
print("--- Collision Distribution by Borough ---")
print(borough_dist)

In [ ]:
# Query 2: Percentage of collisions resulting in injury/death
query_2 = f"""
SELECT 
    SUM(CAST(number_of_persons_injured AS INT) + CAST(number_of_persons_killed AS INT)) as total_casualties,
    COUNT(*) as total_collisions
FROM {database_name}.{table_name}
"""
severity_summary = wr.athena.read_sql_query(query_2, database=database_name)
print("\n--- Severity Summary ---")
print(severity_summary)

In [ ]:
# Query 3: Most common vehicle types involved in collisions
query_3 = f"""
SELECT vehicle_type_code1, COUNT(*) as count
FROM {database_name}.{table_name}
WHERE vehicle_type_code1 IS NOT NULL
GROUP BY vehicle_type_code1
ORDER BY count DESC
LIMIT 10
"""
vehicle_types = wr.athena.read_sql_query(query_3, database=database_name)
print("\n--- Top 10 Vehicle Types Involved ---")
print(vehicle_types)